[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/orientation/conformal_circle/conformal_circle.ipynb)

# A Circle Constructed Twice

This example builds one circle in two ways and reflects it: as the circle through three ordered points on it, and as the circle where a plane cuts a sphere, each surface with a chosen side. Both constructions give the circle an orientation, and a reflection can tell them apart even when their coefficients agree.

In the conformal algebra of space, points are modelled as null antivectors, and spheres and planes as vectors. The circle through three ordered points is their join, `a & b & c`, and intrinsic; the circle cut by a plane and a sphere is their meet, `plane ^ sphere`, and extrinsic. A mirror acts on both alike, `mirror >> circle`, and their orientations decide the sign of the result: reflected in its own plane, the join keeps its sign and the meet changes it. Motions along the circle are generated by extrinsic bivectors, which the intrinsic circle supplies through its dual paired with the sphere at infinity or with a reflecting plane.

In the notation of conformal geometric algebra with points as null vectors, the circle through three points reads as their outer product $P_1 \wedge P_2 \wedge P_3$, and the circle cut by a plane $\pi$ and a sphere $\sigma$ as the dual of the outer product of the two, $(\pi \wedge \sigma)^*$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np
from numga import NumpyContext, stack
from examples.util.animation import save_figure, show_animation
from examples.orientation.conformal_circle import render
from examples.orientation.conformal_circle.core import sphere_samples

In [ ]:
from numga.algebras import CGA3D
ga = CGA3D
mv = NumpyContext(ga).multivector

## 1. The circle through points and the circle cut by surfaces

The unit sphere meets the horizontal plane in the same circle that passes through the three ordered points. Reflection in that plane fixes every circle point and preserves their order. It also fixes the sphere, but reverses the plane's chosen side.

In [ ]:
# Three points of the unit circle in the plane z = 0, at x, at -y and at -x.
points = stack([mv.x.dual() + mv.e.dual(), -mv.y.dual() + mv.e.dual(), -mv.x.dual() + mv.e.dual()])   # [3] Point-
plane = mv.z                                                                # [] Sphere+
sphere = mv.w                                                               # [] Sphere+
mirror = plane                                                              # [] Sphere+
n_samples, n_meridians, n_parallels = 96, 16, 9
angles = np.linspace(0, 2 * np.pi, n_samples + 1)
longitudes = np.linspace(0, 2 * np.pi, n_meridians + 1)
latitudes = np.linspace(-np.pi / 2, np.pi / 2, n_parallels)

In [ ]:
# Joining the ordered points and intersecting the surfaces give the same blade coefficients.
through_points = (points[0] & points[1] & points[2]).normalized()           # [] Path-
from_surfaces = (plane ^ sphere).normalized()                               # [] Circle+

# Each construction reflected; its orientation decides how the mirror acts on it.
reflected_path = mirror >> through_points                                   # [] Path-
reflected_cut = mirror >> from_surfaces                                     # [] Circle+

# The circle and the sphere sampled for display, and the whole scene beside its reflection.
turns = (mv.xy * (angles / 2)).exp()                                        # [samples + 1] Rotor+
curve = turns >> points[0]                                                  # [samples + 1] Point-
surface = sphere_samples(points[0], mv.xy, mv.xz, longitudes, latitudes)    # [parallels, meridians + 1] Point-
vertices = stack([points, mirror >> points])                                # [sides, 3] Point-
curves = stack([curve, mirror >> curve])                                    # [sides, samples + 1] Point-
surfaces = stack([surface, mirror >> surface])                              # [sides, parallels, meridians + 1] Point-
planes = stack([plane, mirror >> plane])                                    # [sides] Sphere+
spheres = stack([sphere, mirror >> sphere])                                 # [sides] Sphere+

np.set_printoptions(precision=3, suppress=True)
print("Through points:\n" + through_points.coefficient_table())
print("From surfaces:\n" + from_surfaces.coefficient_table())
print("Through points, reflected:\n" + reflected_path.coefficient_table())
print("From surfaces, reflected:\n" + reflected_cut.coefficient_table())
save_figure(render.draw_circle(vertices, curves, surfaces, planes, spheres), "orientation_conformal_circle");

The left panel is the construction before reflection; the right is its mirror. The dark circle arrows and three coloured markers keep their order. The orange arrow still points into the sphere, while the teal arrow turns over with the plane's chosen side. Thus the point join remains `+zw`, but the surface intersection becomes `−zw`. Rebuilding either circle from its reflected ingredients gives the corresponding reflected result.

## 2. Motions along the circle

Points can travel along this circle in different ways. On the left, they rotate rigidly about its axis. On the right, a reflection combined with a conformal boost moves them unevenly along the circle, fixing its top and bottom points. This is a [hyperbolic glide reflection](https://hamishtodd1.substack.com/p/hyperbolic-glide-reflections). The intrinsic circle is not itself exponentiated: its dual, paired with the sphere at infinity or a reflecting plane, supplies an extrinsic bivector generator.

In [ ]:
origin = (mv.e.dual() - mv.w.dual()) / 2                                    # [] Point-
infinity = mv.e.dual() + mv.w.dual()                                        # [] Point-
# The sphere at infinity.
horizon = mv.e + mv.w                                                       # [] Sphere+
reflector = mv.x                                                            # [] Sphere+
n_frames, n_radii, n_angles = 64, 40, 96
glide_amplitude = 0.9
duration_ms = 70
phases = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
radii = np.linspace(0, 1, n_radii + 1)
angles = np.linspace(0, 2 * np.pi, n_angles + 1)

In [ ]:
# The sphere at infinity selects the Euclidean rotation about the circle's axis.
axis = (through_points.dual() | horizon).normalized()                       # [] Circle+
rotations = (axis * (-phases / 2)).exp()                                    # [frames] Rotor+

# The reflection commutes with this boost; zero boost still leaves the reflection.
boost = reflector | through_points.dual()                                   # [] Circle+
glides = reflector * (boost * (glide_amplitude * np.sin(phases))).exp()     # [frames] Odd+

# A grid inside the circle makes rigidity and conformal distortion visible: the origin carried along x by each
# radius, the join of x with the point at infinity generating the translation, then turned about the axis.
shifts = ((infinity & mv.x.dual()).dual() * (-radii / 2)).exp()             # [radii + 1] Rotor+
turns = (axis * (angles / 2)).exp()                                         # [angles + 1] Rotor+
grid = turns[None, :] >> (shifts >> origin)[:, None]                        # [radii + 1, angles + 1] Point-
# Each frame, the grid carried by the rotation beside the grid carried by the glide.
frames = (stack([rotations[frame] >> grid, glides[frame] >> grid]) for frame in range(n_frames))
fixed_points = stack([mv.y.dual() + mv.e.dual(), -mv.y.dual() + mv.e.dual()])   # [2] Point-

show_animation(render.animate_circle_motions(frames, fixed_points), "orientation_circle_motions", duration_ms)

The coloured markers follow the same sampled points in both panels. The left grid keeps its shape; the right grid bends and crowds toward one of the two ringed fixed points. The boost sweeps back and forth while the reflection stays present, reversing the marker order. Exponentiating the surface intersection `from_surfaces` instead would fix every point on the circle and move surrounding points around it.